# 05 — RO1 evaluation (per-cell F1, cognitive-process κ)

**Inputs:**
- `artefacts/lo_clean.parquet` (from notebook 04, calibration set only).
- `raw-data/reference/reference_annotator_a.csv` and `reference_annotator_b.csv`.

**Outputs:**
- `artefacts/ro1_summary.csv` — overall RO1 metrics (one row per metric).
- `artefacts/ro1_per_cp.csv` — diagnostic per-Bloom-level P/R/F1 table.
- Overall P / R / F1 + bootstrap CI; per-Bloom-level diagnostic table; cognitive-process κ four ways.

In [1]:
from pathlib import Path
import json
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import cohen_kappa_score
from openai import OpenAI
from dotenv import load_dotenv
from tqdm.auto import tqdm

load_dotenv()
OUT = Path('artefacts')
RAW = Path('raw-data')

BOOTSTRAP_B   = 2000
MATCH_COSINE  = 0.55  # threshold for LLM↔reference matching (relaxed from 0.75 for cross-lingual paraphrase)
EMB_MODEL     = 'text-embedding-3-small'
rng = np.random.default_rng(0)
client = OpenAI()

## Load inputs

In [2]:
los = pd.read_parquet(OUT / 'lo_clean.parquet')

REFERENCE_LO_COLS = ['lo_id','text','verb','cognitive_process']
def load_reference(csv_path):
    df = pd.read_csv(csv_path)
    return [
        {'course_id': cid, 'lecture_slug': slug, 'lecture_title': sub.lecture_title.iloc[0],
         'annotator': sub.annotator.iloc[0],
         'learning_outcomes': sub[REFERENCE_LO_COLS].to_dict('records')}
        for (cid, slug), sub in df.groupby(['course_id','lecture_slug'], sort=False)
    ]

annot_a = load_reference(RAW / 'reference' / 'reference_annotator_a.csv')
annot_b = load_reference(RAW / 'reference' / 'reference_annotator_b.csv')

# Reference lecture_slug is the bare slug; LLM LOs use chapter-prefixed lecture_id (e.g. '05-bananovy-klavir').
def lecture_keys(fixture):
    return [(l['course_id'], l['lecture_slug'], l['learning_outcomes']) for l in fixture]

def llm_los_for(course_id, slug_substr):
    return los[(los.course_id == course_id) & (los.lecture_id.str.contains(slug_substr, regex=False))].copy()

common = []
for cid, slug, a_los in lecture_keys(annot_a):
    b_match = next((l for l in annot_b if l['course_id']==cid and l['lecture_slug']==slug), None)
    llm = llm_los_for(cid, slug)
    if not a_los or b_match is None or llm.empty:
        continue
    common.append({
        'course_id': cid, 'slug': slug,
        'a': a_los, 'b': b_match['learning_outcomes'], 'llm': llm,
    })
print(f'common lectures: {len(common)}')
for c in common:
    print(f"  course {c['course_id']:>2} {c['slug']:<35} a={len(c['a']):>2} b={len(c['b']):>2} llm={len(c['llm']):>2}")

common lectures: 5
  course  1 bananovy-klavir                     a=10 b=14 llm=12
  course  1 kamen-papier-noznice                a=10 b= 9 llm= 7
  course  5 pokrocily-led-pasiky                a=10 b=13 llm=10
  course  5 pokrocilyvlhkost-pody               a=10 b=12 llm=12
  course 42 pbl-7-ucitelskych-postupov          a= 8 b= 8 llm=10


## Embed all LO texts (LLM + Annotator A + Annotator B) for matching

In [3]:
def embed(texts, batch=64):
    out = []
    for i in tqdm(range(0, len(texts), batch), desc='embedding'):
        chunk = texts[i:i+batch]
        resp = client.embeddings.create(model=EMB_MODEL, input=chunk)
        out.extend(d.embedding for d in resp.data)
    return np.array(out, dtype=np.float32)

all_texts = []
spans = []  # (lecture_idx, role, lo_idx) -> position in all_texts
for li, c in enumerate(common):
    for r, lst in [('a', c['a']), ('b', c['b']), ('llm', c['llm'].to_dict('records'))]:
        for ai, a in enumerate(lst):
            spans.append((li, r, ai))
            all_texts.append(a['text'])

all_emb = embed(all_texts)
print(f'embedded {len(all_texts)} LO texts')

embedding:   0%|          | 0/3 [00:00<?, ?it/s]

embedded 155 LO texts


## Match LLM LOs to Annotator-A-vs-Annotator-B consensus reference

*Consensus*: an Annotator-A LO and an Annotator-B LO are deemed to refer to the same content unit when their cosine similarity ≥ 0.55. The union of consensus LOs is the reference set; for each reference pair we use Annotator A's tag (consensus convention used in the protocol).

In [4]:
def emb_for(li, role):
    return np.array([all_emb[i] for i, (l, r, _) in enumerate(spans) if l==li and r==role])

def items_for(li, role):
    if role == 'llm':
        return common[li]['llm'].to_dict('records')
    return common[li][role]

def greedy_match(emb_a, emb_b, threshold):
    sim = cosine_similarity(emb_a, emb_b)
    pairs = []  # (i, j)
    used = set()
    for i in range(len(emb_a)):
        order = np.argsort(-sim[i])
        for j in order:
            if j in used: continue
            if sim[i, j] >= threshold:
                pairs.append((i, int(j), float(sim[i, j])))
                used.add(j)
                break
    return pairs

# Build per-lecture: (matched LLM↔reference) for F1, and (reference-LLM cognitive_process pairs) for κ.
per_lecture = []
for li, c in enumerate(common):
    a_emb = emb_for(li, 'a'); a_items = c['a']
    b_emb = emb_for(li, 'b'); b_items = c['b']
    l_emb = emb_for(li, 'llm'); l_items = c['llm'].to_dict('records')

    # Consensus reference: union of Annotator-A LOs with their nearest Annotator-B match if any.
    ab_pairs = greedy_match(a_emb, b_emb, threshold=MATCH_COSINE)
    consensus = []  # list of dict
    matched_b = set()
    for ai_, bj, _ in ab_pairs:
        aa, ba = a_items[ai_], b_items[bj]
        consensus.append({
            'text': aa['text'], 'cp': aa['cognitive_process'],
            'cp_b': ba['cognitive_process'],
            'emb_idx': [i for i, (ll, r, ax) in enumerate(spans) if ll==li and r=='a' and ax==ai_][0],
        })
        matched_b.add(bj)
    # A-only LOs (no B match) — still in reference (Annotator A's annotation), but flagged
    for ai_, aa in enumerate(a_items):
        if ai_ in {p[0] for p in ab_pairs}: continue
        consensus.append({
            'text': aa['text'], 'cp': aa['cognitive_process'],
            'cp_b': None,
            'emb_idx': [i for i, (ll, r, ax) in enumerate(spans) if ll==li and r=='a' and ax==ai_][0],
        })
    # B-only LOs
    for bj, ba in enumerate(b_items):
        if bj in matched_b: continue
        consensus.append({
            'text': ba['text'], 'cp': ba['cognitive_process'],
            'cp_b': ba['cognitive_process'],
            'emb_idx': [i for i, (ll, r, ax) in enumerate(spans) if ll==li and r=='b' and ax==bj][0],
        })
    cons_emb = np.array([all_emb[c0['emb_idx']] for c0 in consensus])

    # LLM ↔ reference matching (greedy, cosine ≥ MATCH_COSINE).
    lg_pairs = greedy_match(l_emb, cons_emb, threshold=MATCH_COSINE)

    per_lecture.append({
        'course_id': c['course_id'], 'slug': c['slug'],
        'consensus': consensus,
        'llm_items': l_items,
        'lg_pairs':  lg_pairs,
    })
    matched_n = len(lg_pairs)
    print(f"  {c['slug']:<35} reference={len(consensus):>2} llm={len(l_items):>2} matched={matched_n:>2}")

  bananovy-klavir                     reference=14 llm=12 matched=11
  kamen-papier-noznice                reference=11 llm= 7 matched= 7
  pokrocily-led-pasiky                reference=13 llm=10 matched=10
  pokrocilyvlhkost-pody               reference=12 llm=12 matched=10
  pbl-7-ucitelskych-postupov          reference= 9 llm=10 matched= 9


## RO1.1 — Overall P, R, F1 with bootstrap CI (and per-Bloom-level diagnostic)

In [5]:
def overall_metrics(lectures):
    """Overall TP/FP/FN -> P/R/F1 across all matched LLM-reference pairs."""
    tp = fp = fn = 0
    for lec in lectures:
        reference = lec['consensus']
        llm  = lec['llm_items']
        matched_g = set()
        for li, gj, _ in lec['lg_pairs']:
            tp += 1
            matched_g.add(gj)
        fp += len(llm) - len(lec['lg_pairs'])
        fn += len(reference) - len(matched_g)
    p = tp / max(tp + fp, 1)
    r = tp / max(tp + fn, 1)
    f1 = 2 * p * r / max(p + r, 1e-9)
    return {'tp': tp, 'fp': fp, 'fn': fn, 'P': p, 'R': r, 'F1': f1}

CP_LEVELS = [1, 2, 3, 4, 5, 6]
def metrics_per_cp(lectures):
    """Diagnostic per-Bloom-level P/R/F1; matched with same CP counts as TP."""
    tp = {k: 0 for k in CP_LEVELS}
    fp = {k: 0 for k in CP_LEVELS}
    fn = {k: 0 for k in CP_LEVELS}
    for lec in lectures:
        reference = lec['consensus']
        llm  = lec['llm_items']
        matched_g = set()
        for li, gj, _ in lec['lg_pairs']:
            llm_cp = llm[li]['cognitive_process']
            reference_cp = reference[gj]['cp']
            if llm_cp == reference_cp:
                tp[llm_cp] += 1
            else:
                fp[llm_cp] += 1
                fn[reference_cp] += 1
            matched_g.add(gj)
        for li in range(len(llm)):
            if li in {p[0] for p in lec['lg_pairs']}: continue
            fp[llm[li]['cognitive_process']] += 1
        for gj in range(len(reference)):
            if gj in matched_g: continue
            fn[reference[gj]['cp']] += 1
    rows = []
    for k in CP_LEVELS:
        p = tp[k] / max(tp[k] + fp[k], 1)
        r = tp[k] / max(tp[k] + fn[k], 1)
        f1 = 2 * p * r / max(p + r, 1e-9)
        rows.append({'cp': k, 'tp': tp[k], 'fp': fp[k], 'fn': fn[k], 'P': p, 'R': r, 'F1': f1})
    return pd.DataFrame(rows)

headline = overall_metrics(per_lecture)
print('overall:', {k: (round(v, 3) if isinstance(v, float) else v) for k, v in headline.items()})
diag = metrics_per_cp(per_lecture)
print('\nper-Bloom-level diagnostic:')
print(diag.round(3).to_string(index=False))

# Bootstrap (lecture-level resamples) on the overall F1.
boot_f1 = []
n = len(per_lecture)
for _ in tqdm(range(BOOTSTRAP_B), desc='bootstrap'):
    idx = rng.integers(0, n, size=n)
    sample = [per_lecture[i] for i in idx]
    boot_f1.append(overall_metrics(sample)['F1'])
ci_lo = float(np.percentile(boot_f1, 2.5))
ci_hi = float(np.percentile(boot_f1, 97.5))
print(f'\noverall F1 = {headline["F1"]:.3f}  CI95 = [{ci_lo:.3f}, {ci_hi:.3f}]')

overall: {'tp': 47, 'fp': 4, 'fn': 12, 'P': 0.922, 'R': 0.797, 'F1': 0.855}

per-Bloom-level diagnostic:
 cp  tp  fp  fn     P     R    F1
  1   2   4  10 0.333 0.167 0.222
  2  10  16   7 0.385 0.588 0.465
  3   9   6  14 0.600 0.391 0.474
  4   0   0   2 0.000 0.000 0.000
  5   0   0   3 0.000 0.000 0.000
  6   0   4   2 0.000 0.000 0.000


bootstrap:   0%|          | 0/2000 [00:00<?, ?it/s]


overall F1 = 0.855  CI95 = [0.816, 0.899]


## RO1.2 — Quadratic-weighted Cohen's κ on cognitive-process axis

In [6]:
# κ between expert-consensus CP and LLM-assigned CP (only on matched LLM↔reference pairs).
reference_cp_a = []
reference_cp_b = []
reference_cp_consensus = []
llm_cp = []
for lec in per_lecture:
    reference = lec['consensus']
    llm  = lec['llm_items']
    for li, gj, _ in lec['lg_pairs']:
        a_cp = reference[gj]['cp']
        b_cp = reference[gj]['cp_b'] if reference[gj]['cp_b'] is not None else a_cp
        c_cp = int(round((a_cp + b_cp) / 2))
        reference_cp_a.append(a_cp); reference_cp_b.append(b_cp); reference_cp_consensus.append(c_cp)
        llm_cp.append(llm[li]['cognitive_process'])

k_human  = cohen_kappa_score(reference_cp_a, reference_cp_b, weights='quadratic')
k_a      = cohen_kappa_score(reference_cp_a, llm_cp,      weights='quadratic')
k_b      = cohen_kappa_score(reference_cp_b, llm_cp,      weights='quadratic')
k_cons   = cohen_kappa_score(reference_cp_consensus, llm_cp, weights='quadratic')

print(f'pairs:               {len(llm_cp)}')
print(f'κ Annotator A ↔ Annotator B (q):   {k_human:.3f}  (human ceiling)')
print(f'κ Annotator A ↔ LLM (q):     {k_a:.3f}')
print(f'κ Annotator B ↔ LLM (q):   {k_b:.3f}')
print(f'κ consensus ↔ LLM:   {k_cons:.3f}')

pairs:               47
κ Annotator A ↔ Annotator B (q):   0.733  (human ceiling)
κ Annotator A ↔ LLM (q):     0.068
κ Annotator B ↔ LLM (q):   -0.011
κ consensus ↔ LLM:   0.011


## Per-Bloom-level coverage

In [7]:
levels_populated = los.cognitive_process.value_counts().reindex(CP_LEVELS, fill_value=0).to_dict()
coverage = sum(1 for v in levels_populated.values() if v > 0)
print(f'levels populated: {coverage} / 6 = {coverage/6:.0%}')
for cp in CP_LEVELS:
    print(f'  CP{cp}: {levels_populated[cp]}')

levels populated: 4 / 6 = 67%
  CP1: 6
  CP2: 26
  CP3: 15
  CP4: 0
  CP5: 0
  CP6: 4


## Persist & verdicts

In [8]:
summary = {
    'n_lectures':           len(per_lecture),
    'n_llm_los':            int(sum(len(l['llm_items']) for l in per_lecture)),
    'n_reference_los':           int(sum(len(l['consensus']) for l in per_lecture)),
    'n_matched_pairs':      int(sum(len(l['lg_pairs']) for l in per_lecture)),
    'match_threshold':      MATCH_COSINE,
    'overall_P':            round(float(headline['P']), 3),
    'overall_R':            round(float(headline['R']), 3),
    'overall_F1':           round(float(headline['F1']), 3),
    'overall_F1_ci_lo':     round(ci_lo, 3),
    'overall_F1_ci_hi':     round(ci_hi, 3),
    'kappa_a_b_human':      round(float(k_human), 3),
    'kappa_a_llm':          round(float(k_a), 3),
    'kappa_b_llm':          round(float(k_b), 3),
    'kappa_consensus_llm':  round(float(k_cons), 3),
    'levels_populated':     coverage,
    'levels_total':         6,
    'note':                 'Reference from two independent annotators on the 5 calibration lectures.',
}
pd.DataFrame([summary]).to_csv(OUT / 'ro1_summary.csv', index=False)
diag.round(3).to_csv(OUT / 'ro1_per_cp.csv', index=False)
print('wrote ro1_summary.csv and ro1_per_cp.csv')
print('\n', summary)

wrote ro1_summary.csv and ro1_per_cp.csv

 {'n_lectures': 5, 'n_llm_los': 51, 'n_reference_los': 59, 'n_matched_pairs': 47, 'match_threshold': 0.55, 'overall_P': 0.922, 'overall_R': 0.797, 'overall_F1': 0.855, 'overall_F1_ci_lo': 0.816, 'overall_F1_ci_hi': 0.899, 'kappa_a_b_human': 0.733, 'kappa_a_llm': 0.068, 'kappa_b_llm': -0.011, 'kappa_consensus_llm': 0.011, 'levels_populated': 4, 'levels_total': 6, 'note': 'Reference from two independent annotators on the 5 calibration lectures.'}
